# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

**TODO: 팀에서 준비한 CSV 파일 경로를 입력하세요**

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [2]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "iot_common_guidelines": "../datasets/1_parent_iot_common_guidelines.csv",
    "home_iot_controls": "../datasets/2_child_home_iot_controls.csv",
    "stm32_debug_topics": "../datasets/3_child_stm32_debug_topics.csv",
    "esp32h2_features": "../datasets/4_child_esp32h2_features.csv",
    "iot_security_agencies_parent": "../datasets/5_parent_iot_security_agencies.csv",
    "iot_security_incidents_child": "../datasets/6_child_iot_security_incidents.csv"
}


# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 iot_common_guidelines 테이블

행 수: 15
컬럼: ['guideline_id', 'principle_id', 'principle_name', 'lifecycle_stage', 'guideline_code', 'guideline_name', 'page_start', 'domain', 'description', 'design_focus', 'security_focus', 'easy_explanation']

첫 5개 행:
   guideline_id  principle_id                     principle_name  \
0             1             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
1             2             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
2             3             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
3             4             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
4             5             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   

  lifecycle_stage guideline_code                guideline_name  page_start  \
0           설계·개발            G01     IoT 장치 특성을 고려한 보안 서비스 경량화          20   
1           설계·개발            G02  접근권한 관리·인증·종단간 통신 보안·데이터 암호화          31   
2           설계·개발            G03         소프트웨어·하드웨어 보안기술 적용 검토          33   
3           설계·개발            G

## 2. 데이터 탐색 및 통계

**TODO: 팀 데이터에 맞는 탐색 쿼리를 작성하세요**

In [3]:
for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    df.info()

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()

    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # 고유값 개수
    print("\n[고유값 개수]")
    for col in df.columns:
        print(f"{col}: {df[col].nunique()}개")

    # 카테고리별 데이터 분포
    print("\n[카테고리 분포]")
    for col in df.select_dtypes(include=['object']).columns:
        print(f"\n[{col}]")
        print(df[col].value_counts())


📊 iot_common_guidelines 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   guideline_id      15 non-null     int64
 1   principle_id      15 non-null     int64
 2   principle_name    15 non-null     str  
 3   lifecycle_stage   15 non-null     str  
 4   guideline_code    15 non-null     str  
 5   guideline_name    15 non-null     str  
 6   page_start        15 non-null     int64
 7   domain            15 non-null     str  
 8   description       15 non-null     str  
 9   design_focus      15 non-null     str  
 10  security_focus    15 non-null     str  
 11  easy_explanation  15 non-null     str  
dtypes: int64(3), str(9)
memory usage: 8.7 KB

[결측치]
결측치 없음

[고유값 개수]
guideline_id: 15개
principle_id: 7개
principle_name: 7개
lifecycle_stage: 3개
guideline_code: 15개
guideline_name: 15개
page_start: 15개
domain: 15개
description: 15개
design_focus

C:\Users\khm35\AppData\Local\Temp\ipykernel_21656\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\khm35\AppData\Local\Temp\ipykernel_21656\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

## 3. Supabase PostgreSQL 연결

In [4]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

C:\Users\khm35\AppData\Local\Temp\ipykernel_21656\1442992373.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['1_parent_iot_common_guidelines', '2_child_home_iot_controls', '3_child_stm32_debug_topics', '4_child_esp32h2_features', 'categories', 'departments', 'forms', 'iot_security_agencies_parent', 'iot_security_incidents_child', 'office_floors', 'organizations', 'topics']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [5]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE "1_parent_iot_common_guidelines" (
	guideline_id BIGINT, 
	principle_id BIGINT, 
	principle_name TEXT, 
	lifecycle_stage TEXT, 
	guideline_code TEXT, 
	guideline_name TEXT, 
	page_start BIGINT, 
	domain TEXT, 
	description TEXT, 
	design_focus TEXT, 
	security_focus TEXT, 
	easy_explanation TEXT
)

/*
3 rows from 1_parent_iot_common_guidelines table:
guideline_id	principle_id	principle_name	lifecycle_stage	guideline_code	guideline_name	page_start	domain	description	design_focus	security_focus	easy_explanation
1	1	정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계	설계·개발	G01	IoT 장치 특성을 고려한 보안 서비스 경량화	20	하드웨어_설계	프로세서 성능, 메모리, 입출력장치, 소비전력 등 장치 자원 수준을 고려하여 필요한 보안 기능을 경량화해 구현	MCU 성능, 메모리, 입출력장치, 소비전력 등 기기 자원을 먼저 확인	기기 성능을 넘지 않는 범위에서 필요한 보안 기능을 적용	기기의 성능과 전력에 맞춰 하드웨어와 보안 기능의 크기를 정하는 기준
2	1	정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계	설계·개발	G02	접근권한 관리·인증·종단간 통신 보안·데이터 암호화	31	인증_통신_암호화	IoT 서비스 환경에 맞는 접근권한, 인증, 통신 보호, 데이터 암호화 방안을 제공	기기와 외부 장치·서버가 연결되는 통신 경로를 확인	허가된 사용자·장치만 접근하게 하고 통신과 데이터를

## 6. SQL 쿼리 테스트

**TODO: 팀 데이터에 맞는 SQL 쿼리를 작성하여 테스트하세요**

In [6]:
# 보안 사고 유형별 신고 기관 및 대처 방법 통합 조회
# 부모: iot_security_agencies_parent
# 자식: iot_security_incidents_child

query = """
SELECT
    i.incident_type,
    i.example_symptom,
    i.recommended_action,
    a.agency_name,
    a.organization,
    a.phone,
    a.contact_method
FROM "iot_security_incidents_child" AS i
INNER JOIN "iot_security_agencies_parent" AS a
    ON i.agency_id = a.agency_id
ORDER BY
    a.agency_id,
    i.incident_id;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    i.incident_type,
    i.example_symptom,
    i.recommended_action,
    a.agency_name,
    a.organization,
    a.phone,
    a.contact_method
FROM "iot_security_incidents_child" AS i
INNER JOIN "iot_security_agencies_parent" AS a
    ON i.agency_id = a.agency_id
ORDER BY
    a.agency_id,
    i.incident_id;


결과:
[('해킹 의심', 'IoT 앱·서버·클라우드에 모르는 로그인 또는 비정상 접속 발생', '우선 118에 상담 후 계정 비밀번호 변경 및 접속기록 보존', 'KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담'), ('악성코드/바이러스', 'IoT 관리 PC·서버·게이트웨이에서 악성 프로그램 또는 이상 동작 발견', '네트워크 격리 후 118 상담 및 로그·파일 보존', 'KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담'), ('계정 탈취', 'IoT 관리자 계정 비밀번호가 바뀌거나 권한이 탈취됨', '비밀번호 변경·세션 종료 후 118 상담', 'KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담'), ('개인정보 침해', 'IoT 카메라 영상, 위치정보, 사용자 계정정보 등이 외부로 유출된 것으로 의심', '118 개인정보 상담 이용 및 관련 기록 보존', 'KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담'), ('스미싱/피싱', 'IoT 서비스 사칭 문자·링크로 계정정보 입력 또는 악성앱 설치가 의심됨', '118 상담 후 비밀번호 변경 및 악성앱 점검', 'KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담'), ('

In [7]:
# TODO: JOIN 쿼리 작성
# 부모 테이블(신고 기관)과 자식 테이블(사고 유형)을 연결하고,
# 기존 보안 가이드라인과 홈가전 보안 항목도 함께 통합 조회

join_query = """
SELECT
    a.agency_name,
    '보안 사고 신고' AS source_type,
    i.incident_type AS item_name,
    i.example_symptom AS detail,
    i.recommended_action AS action_or_purpose
FROM "iot_security_incidents_child" i
INNER JOIN "iot_security_agencies_parent" a
    ON i.agency_id = a.agency_id

UNION ALL

SELECT
    g.guideline_name,
    '보안 가이드라인' AS source_type,
    h.control_name AS item_name,
    h.easy_explanation AS detail,
    h.security_purpose AS action_or_purpose
FROM "2_child_home_iot_controls" h
INNER JOIN "1_parent_iot_common_guidelines" g
    ON h.parent_guideline_id = g.guideline_id

ORDER BY source_type, item_name;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    a.agency_name,
    '보안 사고 신고' AS source_type,
    i.incident_type AS item_name,
    i.example_symptom AS detail,
    i.recommended_action AS action_or_purpose
FROM "iot_security_incidents_child" i
INNER JOIN "iot_security_agencies_parent" a
    ON i.agency_id = a.agency_id

UNION ALL

SELECT
    g.guideline_name,
    '보안 가이드라인' AS source_type,
    h.control_name AS item_name,
    h.easy_explanation AS detail,
    h.security_purpose AS action_or_purpose
FROM "2_child_home_iot_controls" h
INNER JOIN "1_parent_iot_common_guidelines" g
    ON h.parent_guideline_id = g.guideline_id

ORDER BY source_type, item_name;


결과:
[('접근권한 관리·인증·종단간 통신 보안·데이터 암호화', '보안 가이드라인', 'IoT 제품 간 상호인증', '기기끼리 통신할 때 서로 진짜 장치인지 확인', '가짜 장치가 정상 장치인 것처럼 연결되는 것을 줄임'), ('로그기록 저장·관리', '보안 가이드라인', '감사기록', '기기에서 중요한 일이 일어난 기록을 남김', '사고 발생 시 원인과 행동을 추적'), ('개인정보보호정책 및 보호조치 마련', '보안 가이드라인', '개인정보 보호', '필요한 개인정보만 쓰고 안전하게 보관', '불필요한 개인정보 수집과 유출을 줄임'), ('다양한 하드웨어 보안기법 적용', '보안 가이드라인', '내부 입출력 포트 비활성화', '개발

In [8]:
# TODO: 집계(Aggregation) 쿼리 작성
# 신고 기관별 대응 가능한 사고 유형 수 + 보안 가이드라인별 점검 항목 수

aggregation_query = """
SELECT
    a.agency_name AS 항목,
    a.organization AS 소속,
    a.phone AS 연락처,
    COUNT(i.incident_id) AS 대응_사고_유형_수
FROM "iot_security_agencies_parent" a
LEFT JOIN "iot_security_incidents_child" i
    ON i.agency_id = a.agency_id
GROUP BY a.agency_name, a.organization, a.phone
ORDER BY 대응_사고_유형_수 DESC;
"""

print("실행 쿼리 1: 신고 기관별 대응 가능한 사고 유형 수")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")


aggregation_query2 = """
SELECT
    g.guideline_name AS 가이드라인,
    g.lifecycle_stage AS 수명주기_단계,
    COUNT(DISTINCT h.home_control_id) AS 홈가전_보안항목_수
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_name, g.lifecycle_stage
HAVING COUNT(DISTINCT h.home_control_id) > 0
ORDER BY 홈가전_보안항목_수 DESC;
"""

print("\n\n실행 쿼리 2: 보안 가이드라인별 홈가전 점검 항목 수")
print(aggregation_query2)
print("\n결과:")

try:
    result2 = db.run(aggregation_query2)
    print(result2)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리 1: 신고 기관별 대응 가능한 사고 유형 수

SELECT
    a.agency_name AS 항목,
    a.organization AS 소속,
    a.phone AS 연락처,
    COUNT(i.incident_id) AS 대응_사고_유형_수
FROM "iot_security_agencies_parent" a
LEFT JOIN "iot_security_incidents_child" i
    ON i.agency_id = a.agency_id
GROUP BY a.agency_name, a.organization, a.phone
ORDER BY 대응_사고_유형_수 DESC;


결과:
[('KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, 5), ('KISA 보호나라·KrCERT/CC', '한국인터넷진흥원(KISA)', 118, 4), ('경찰청 사이버범죄 신고시스템(ECRM)', '경찰청', 182, 3)]


실행 쿼리 2: 보안 가이드라인별 홈가전 점검 항목 수

SELECT
    g.guideline_name AS 가이드라인,
    g.lifecycle_stage AS 수명주기_단계,
    COUNT(DISTINCT h.home_control_id) AS 홈가전_보안항목_수
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_name, g.lifecycle_stage
HAVING COUNT(DISTINCT h.home_control_id) > 0
ORDER BY 홈가전_보안항목_수 DESC;


결과:
[('다양한 하드웨어 보안기법 적용', '설계·개발', 4), ('접근권한 관리·인증·종단간 통신 보안·데이터 암호화', '설계·개발', 4), ('소프트웨어 취약점 점검 및 보안패치 방안 구현', '

## 7. Text2SQL 함수 구현

**TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요**

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
import re

llm = init_chat_model("gpt-5.4-mini")

# 개선 3: SQL 안전성 검증 (SELECT/CTE 이외의 위험한 구문 차단)
FORBIDDEN_KEYWORDS = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", "CREATE", "GRANT", "REVOKE"]

def validate_sql(sql: str) -> None:
    """생성된 SQL이 조회(SELECT/WITH)만 수행하는지 검증. 위반 시 ValueError 발생"""
    stripped = sql.strip().rstrip(";").strip()
    if not re.match(r"^(SELECT|WITH)\b", stripped, re.IGNORECASE):
        raise ValueError(f"SELECT/WITH로 시작하는 조회 쿼리만 허용됩니다. 생성된 SQL: {sql}")

    for kw in FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{kw}\b", stripped, re.IGNORECASE):
            raise ValueError(f"허용되지 않는 SQL 키워드가 포함되어 있습니다: {kw}")


def text_to_sql(question: str, db: SQLDatabase, error_feedback: str = None) -> str:
    """
    자연어 질문을 SQL로 변환
    error_feedback: 이전 시도가 실패했을 때, 그 오류 메시지를 함께 전달하여 재생성을 유도
    """
    system_prompt = f"""
당신은 IoT 디바이스 보안 점검 도우미의 데이터베이스 전문가입니다.
일반 사용자의 자연어 질문을 PostgreSQL SELECT 쿼리로 변환하세요.

데이터베이스 스키마:
{db.table_info}

<데이터베이스 설명>

- "iot_security_agencies_parent":
  보안 사고 신고 기관 정보입니다. (3개 기관)
  agency_id가 기본 식별자이며,
  agency_name(기관명), organization(소속), phone(전화번호),
  contact_method(연락 방법), main_role(주요 역할), source_url(홈페이지) 정보를 포함합니다.
  - KISA 118 상담센터: 해킹, 악성코드, 개인정보 침해, 스미싱 등 상담
  - KISA 보호나라/KrCERT: 랜섬웨어, DDoS, 서버 침입 등 침해사고 신고
  - 경찰청 ECRM: 사이버범죄 피해 신고, 수사 요청

- "iot_security_incidents_child":
  보안 사고 유형별 증상과 대처 방법입니다. (12개 유형)
  agency_id로 "iot_security_agencies_parent".agency_id를 참조합니다.
  incident_type(사고 유형), example_symptom(증상 예시),
  recommended_action(권장 대처 방법) 정보를 포함합니다.

- "1_parent_iot_common_guidelines":
  IoT 공통 보안 가이드의 상위 기준 테이블입니다.
  guideline_id가 기본 식별자이며,
  principle_name, lifecycle_stage, guideline_name, domain,
  description, easy_explanation 등의 정보를 포함합니다.

- "2_child_home_iot_controls":
  홈·가전 IoT 보안가이드의 세부 보안 점검 항목입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  control_name, control_category, hardware_or_security,
  implementation_summary, easy_explanation, page_start 등의 정보를 포함합니다.

- "3_child_stm32_debug_topics":
  STM32 MCU의 디버깅·보안 기능 정보입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.

- "4_child_esp32h2_features":
  ESP32-H2의 하드웨어 및 보안 기능 정보입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.

</데이터베이스 설명>

<테이블 관계>

"iot_security_agencies_parent".agency_id
    ← "iot_security_incidents_child".agency_id

"1_parent_iot_common_guidelines".guideline_id
    ← "2_child_home_iot_controls".parent_guideline_id

"1_parent_iot_common_guidelines".guideline_id
    ← "3_child_stm32_debug_topics".parent_guideline_id

"1_parent_iot_common_guidelines".guideline_id
    ← "4_child_esp32h2_features".parent_guideline_id

</테이블 관계>

<자주 묻는 질문 유형>

1. 신고/도움 요청 관련:
   "해킹당하면 어디에 신고해?" → iot_security_agencies_parent 조회
   "어떤 사고 유형이 있어?" → iot_security_incidents_child 조회
   "웹캠 해킹 신고 방법" → agencies + incidents JOIN

2. 보안 점검 관련:
   "보안 점검 항목 알려줘" → 2_child_home_iot_controls 조회
   "인증 관련 보안 항목" → control_category로 필터링
   "보안 가이드라인 목록" → 1_parent_iot_common_guidelines 조회

3. 통합 조회:
   "해킹 대처 방법과 보안 점검 항목 같이 알려줘" → 여러 테이블 JOIN 또는 UNION

</자주 묻는 질문 유형>

<개선 1: SQL 예시 (few-shot)>

질문: "해킹당하면 어디에 신고해?"
SQL:
SELECT agency_name, organization, phone, contact_method, main_role
FROM "iot_security_agencies_parent"
ORDER BY agency_id;

질문: "신고 기관별로 대응 가능한 사고 유형이 몇 개인지 알려줘"
SQL:
SELECT a.agency_name, COUNT(i.incident_id) AS incident_count
FROM "iot_security_agencies_parent" a
LEFT JOIN "iot_security_incidents_child" i ON a.agency_id = i.agency_id
GROUP BY a.agency_name
ORDER BY incident_count DESC;

질문: "가이드라인별 홈가전 보안 점검 항목 수를 많은 순서로 알려줘"
SQL:
WITH control_counts AS (
    SELECT parent_guideline_id, COUNT(*) AS cnt
    FROM "2_child_home_iot_controls"
    GROUP BY parent_guideline_id
)
SELECT g.guideline_name, c.cnt,
       RANK() OVER (ORDER BY c.cnt DESC) AS rank
FROM control_counts c
JOIN "1_parent_iot_common_guidelines" g ON g.guideline_id = c.parent_guideline_id
ORDER BY c.cnt DESC;

</개선 1: SQL 예시 (few-shot)>

규칙:
- PostgreSQL 문법을 사용하세요.
- SELECT 쿼리만 생성하세요. INSERT, UPDATE, DELETE, DROP, ALTER는 금지합니다.
- 테이블명에 숫자나 특수문자가 있으므로 반드시 큰따옴표(" ")로 감싸세요.
- 신고 기관/사고 유형 관련 질문은 "iot_security_agencies_parent"와
  "iot_security_incidents_child"를 우선 사용하세요.
- 홈캠, 도어락, 공유기 등 홈·가전 IoT 보안 질문은
  "2_child_home_iot_controls"를 우선 사용하세요.
- 쉬운 설명이 필요한 경우 easy_explanation 컬럼을 활용하세요.
- 결과가 너무 많을 가능성이 있으면 LIMIT을 사용하세요.
- SQL 코드만 반환하세요.
- 설명은 반환하지 마세요.
- 코드 블록(```) 없이 순수 SQL만 반환하세요.
- 반드시 세미콜론(;)으로 끝내세요.

사용 가능한 SQL 문법:
- JOIN (INNER, LEFT, RIGHT, FULL)
- UNION, UNION ALL
- GROUP BY, HAVING
- COUNT, SUM, AVG, MIN, MAX
- 서브쿼리
- WHERE, ORDER BY, LIMIT
- CTE (WITH 절)
- 윈도우 함수
"""

    human_content = question
    if error_feedback:
        human_content = f"""이전에 생성한 SQL이 아래 오류로 실패했습니다. 오류를 참고하여 올바른 SQL을 다시 생성하세요.

오류 메시지: {error_feedback}

원래 질문: {question}
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_content)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료 (few-shot 예시 + SQL 안전성 검증 포함)")

✓ Text2SQL 함수 준비 완료 (few-shot 예시 + SQL 안전성 검증 포함)


## 8. Text2SQL 테스트

**TODO: 팀 데이터에 맞는 자연어 질문으로 테스트하세요**

In [10]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "스마트 기기가 해킹당한 것 같으면 어디에 신고하고, 어떻게 대처해야 하나요?"

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: 스마트 기기가 해킹당한 것 같으면 어디에 신고하고, 어떻게 대처해야 하나요?


생성된 SQL:
SELECT 
    a.agency_name,
    a.organization,
    a.phone,
    a.contact_method,
    a.main_role,
    i.incident_type,
    i.example_symptom,
    i.recommended_action
FROM "iot_security_incidents_child" i
JOIN "iot_security_agencies_parent" a
    ON i.agency_id = a.agency_id
ORDER BY a.agency_id, i.incident_id;


실행 결과:
[('KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담', '해킹·바이러스, 개인정보 침해, 스미싱 등 사이버 보안 상담', '해킹 의심', 'IoT 앱·서버·클라우드에 모르는 로그인 또는 비정상 접속 발생', '우선 118에 상담 후 계정 비밀번호 변경 및 접속기록 보존'), ('KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담', '해킹·바이러스, 개인정보 침해, 스미싱 등 사이버 보안 상담', '악성코드/바이러스', 'IoT 관리 PC·서버·게이트웨이에서 악성 프로그램 또는 이상 동작 발견', '네트워크 격리 후 118 상담 및 로그·파일 보존'), ('KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담', '해킹·바이러스, 개인정보 침해, 스미싱 등 사이버 보안 상담', '계정 탈취', 'IoT 관리자 계정 비밀번호가 바뀌거나 권한이 탈취됨', '비밀번호 변경·세션 종료 후 118 상담'), ('KISA 118 상담센터', '한국인터넷진흥원(KISA)', 118, '전화 상담', '해킹·바이러스, 개인정보 침해, 스미싱 등 사이버 보안 상담', '개인정보 침해', 'IoT 카메라 영상, 위치

## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

**TODO: 답변 생성 프롬프트를 팀 데이터에 맞게 수정하세요**

In [ ]:
import pandas as pd
from sqlalchemy import text as sql_text
from datetime import datetime

query_history = []

def query_database(question: str, db: SQLDatabase, max_retries: int = 2) -> str:
    sql = None
    result_df = None
    error_msg = None

    for attempt in range(1, max_retries + 2):
        print(f"[1] SQL 생성 중... (시도 {attempt})")
        sql = text_to_sql(question, db, error_feedback=error_msg)
        print(f"    {sql}\n")

        try:
            validate_sql(sql)
            print(f"[2] SQL 실행 중...")
            result_df = pd.read_sql(sql_text(sql.rstrip(";")), db._engine)
            print(f"    실행 완료 ({len(result_df)}행)\n")
            error_msg = None
            break
        except Exception as e:
            error_msg = str(e)
            print(f"    ✗ 실패: {error_msg}\n")
            if attempt == max_retries + 1:
                query_history.append({
                    "timestamp": datetime.now().isoformat(timespec="seconds"),
                    "question": question,
                    "sql": sql,
                    "success": False,
                    "error": error_msg,
                })
                return f"SQL 실행에 반복적으로 실패했습니다. 마지막 오류: {error_msg}"

    if result_df is not None and not result_df.empty:
        print("[결과 표]")
        print(result_df.to_string(index=False))
        print()

    query_history.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "question": question,
        "sql": sql,
        "success": True,
        "row_count": len(result_df) if result_df is not None else 0,
    })

    print(f"[3] 답변 생성 중...")

    system_prompt = """
당신은 'IoT 디바이스 보안 점검 도우미'입니다.
조회된 데이터를 바탕으로 일반 사용자가 바로 행동할 수 있도록 쉽고 친절하게 답변하세요.

사용자는 IT 전문가가 아닌 일반인입니다.
웹캠, 스마트 도어락, 공유기 같은 스마트 기기의 보안이 걱정되어 질문하는 사람입니다.

[답변 유형별 안내]
1. 신고/도움 요청 질문일 때:
   - 어디에 연락해야 하는지 기관명, 전화번호, 연락 방법을 명확하게 안내
   - 사고 유형에 따라 어떤 기관이 적합한지 구분해서 설명
   - 신고 전에 준비할 것(증거 보존, 로그 저장 등)을 쉽게 안내
   - 긴급한 경우와 그렇지 않은 경우를 구분

2. 보안 점검 질문일 때:
   - 점검 항목을 체크리스트처럼 보기 쉽게 정리
   - 각 항목이 왜 중요한지 한 줄로 쉽게 설명
   - 가장 중요한 항목부터 순서대로 안내
   - 비전문가도 직접 확인할 수 있는 방법 위주로 설명

3. 이상 증상 관련 질문일 때:
   - 해당 증상이 어떤 사고 유형에 해당하는지 알려주기
   - 즉시 할 수 있는 대처 방법을 순서대로 안내
   - 상황이 심각할 수 있는 경우 신고 기관도 함께 안내

[답변 규칙]
- 조회 결과에 있는 내용을 중심으로 답변하세요.
- 조회 결과에 없는 전화번호, 기관명, 구체적인 수치를 만들어내지 마세요.
- 전문용어는 최대한 사용하지 마세요. 꼭 필요하면 괄호 안에 쉬운 설명을 붙이세요.
  예: "기기 소프트웨어(펌웨어)를 업데이트하세요."
- 기능 이름만 나열하지 말고 "왜 필요한지", "안 하면 어떤 위험이 있는지"를 설명하세요.
- SQL이나 데이터베이스라는 표현은 답변에서 언급하지 마세요.
- 정보가 부족하면 "현재 자료에서는 이 부분을 정확하게 안내하기 어렵습니다."라고 안내하세요.
- 한 문장을 짧고 명확하게 작성하세요.
- 답변 마지막에 "더 궁금한 점이 있으면 언제든 물어보세요." 같은 친근한 마무리를 넣으세요.
"""

    result_text = result_df.to_string(index=False) if result_df is not None else "(결과 없음)"

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result_text}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료 (재시도 / 검증 / 표 포맷팅 / 히스토리 포함)")

✓ 완전한 Text2SQL 시스템 준비 완료 (재시도 / 검증 / 표 포맷팅 / 히스토리 포함)


In [12]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "웹캠이 해킹당한 것 같은데 어디에 신고하고 어떻게 대처해야 하나요?"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 웹캠이 해킹당한 것 같은데 어디에 신고하고 어떻게 대처해야 하나요?


[1] SQL 생성 중... (시도 1)
    SELECT a.agency_name,
       a.organization,
       a.phone,
       a.contact_method,
       a.main_role,
       i.incident_type,
       i.example_symptom,
       i.recommended_action
FROM "iot_security_incidents_child" i
JOIN "iot_security_agencies_parent" a
  ON i.agency_id = a.agency_id
WHERE i.incident_type LIKE '%해킹%'
   OR i.example_symptom LIKE '%웹캠%'
   OR i.example_symptom LIKE '%비정상 접속%'
   OR i.example_symptom LIKE '%모르는 로그인%'
ORDER BY a.agency_id, i.incident_id;

[2] SQL 실행 중...
    실행 완료 (1행)

[결과 표]


ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [ ]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 신고 조회, 사고 유형, 보안 점검, 집계, 통합 조회 등 다양한 유형 포함

questions = [
    # 신고 기관 조회
    "IoT 기기가 해킹당하면 어디에 신고할 수 있나요?",

    # 사고 유형별 대처
    "웹캠에서 영상이 유출된 것 같은데 어떻게 대처해야 하나요?",

    # 보안 점검 항목 조회
    "홈가전 IoT 기기의 보안 점검 항목을 알려주세요.",

    # 집계 - 기관별 사고 유형 수
    "신고 기관별로 대응할 수 있는 사고 유형이 각각 몇 개인가요?",

    # 특정 사고 유형 조회
    "랜섬웨어에 걸리면 어디에 신고하고 어떻게 해야 하나요?",

    # 보안 가이드라인 조회
    "IoT 공통 보안 가이드라인 중 인증과 관련된 항목을 알려주세요.",

    # 통합 조회 - 증상 + 신고 + 대처
    "공유기가 갑자기 느려졌는데 해킹인지 확인하는 방법과 신고 절차를 알려주세요.",
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: IoT 기기가 해킹당하면 어디에 신고할 수 있나요?

[1] SQL 생성 중...
    SELECT
    agency_name,
    organization,
    phone,
    contact_method,
    main_role,
    source_url
FROM "iot_security_agencies_parent"
ORDER BY agency_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


IoT 기기가 해킹당했다면, 아래 기관에 신고하거나 상담할 수 있습니다.

## 1) 가장 먼저 연락하기 좋은 곳
### KISA 118 상담센터
- **기관:** 한국인터넷진흥원(KISA)
- **전화:** **118**
- **방법:** 전화 상담
- **어떤 경우에 좋나:**  
  해킹, 바이러스, 개인정보 침해, 스미싱 같은 보안 문제를 상담할 때 적합합니다.

### KISA 보호나라 · KrCERT/CC
- **기관:** 한국인터넷진흥원(KISA)
- **전화:** **118**
- **방법:** 온라인 침해사고 신고 / 전화상담
- **어떤 경우에 좋나:**  
  랜섬웨어, DDoS, 그 밖의 해킹 같은 **침해사고 신고**와 **기술 지원**이 필요할 때 적합합니다.

## 2) 피해가 범죄 같다면
### 경찰청 사이버범죄 신고시스템(ECRM)
- **기관:** 경찰청
- **전화:** **182**
- **방법:** 온라인 신고·상담 / 전화 민원상담
- **어떤 경우에 좋나:**  
  해킹 피해가 **범죄 피해**로 보이거나, **수사 요청**이 필요할 때 적합합니다.

## 신고 전에 해두면 좋은 것
- **화면 사진을 찍어두세요.**  
  이상한 로그인, 설정 변경, 경고 메시지를 남겨두면 도움이 됩니다.
- **기기 상태를 그대로 보관하세요.**  
  바로 초기화하면 원인 확인이 어려울 수 있습니다.
- **언제부터 이상했는지 적어두세요.**  
  피해 시점을 알아야 상담과 신고에 도움이 됩니다.
- **계정 비밀번호를 먼저 바꾸세요.**  
  같은 비밀번호를 쓰는 다른 서비스도 함께 위험할 수 있습니다.

## 어떻게 선택하면 좋을까요?
- **기기 해킹이 의심되고 우선 상담이 필요하다면:** **KISA 118**
- **해킹으로 인한 침해사고 신고와 기술 지원이 필요하다면:** **KISA 보호나라·KrCERT/CC**
- **범죄 피해로 신고하고 수사 요청을 하고 싶다면:** **경찰청 ECRM 182**

현재 자료에는 이 기관들 외의 추가 안내는 없습니다.  
더 궁금한 점이 있으면 언제든 물어보세요.


질문: 웹캠에서 영상이 유출된 것 같은데 어떻게 대처해야 하나요?

[1] SQL 생성 중...
    SELECT 
  a.agency_name,
  a.organization,
  a.phone,
  a.contact_method,
  a.main_role,
  i.incident_type,
  i.example_symptom,
  i.recommended_action
FROM "iot_security_agencies_parent" a
JOIN "iot_security_incidents_child" i
  ON a.agency_id = i.agency_id
WHERE i.incident_type LIKE '%해킹%'
   OR i.example_symptom LIKE '%비정상 접속%'
   OR i.example_symptom LIKE '%영상%'
   OR i.example_symptom LIKE '%유출%'
ORDER BY a.agency_id, i.incident_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


웹캠 영상이 유출된 것 같다면, **개인정보 침해 또는 해킹 의심** 상황으로 볼 수 있습니다.  
현재 자료 기준으로는 **KISA 118 상담센터(118)** 에 먼저 상담하는 것이 가장 적절합니다.

## 지금 바로 할 일
1. **KISA 118에 전화 상담하기**
   - 전화번호: **118**
   - 이유: 해킹 의심, 개인정보 유출 의심을 함께 상담할 수 있습니다.

2. **계정 비밀번호 바로 변경하기**
   - 웹캠 앱, 연결된 이메일, 클라우드 계정 비밀번호를 바꾸세요.
   - 왜 필요한가요?  
     이미 계정이 넘어갔을 수 있어서, 비밀번호를 바꾸지 않으면 계속 접속될 수 있습니다.

3. **접속 기록과 이상 흔적 보존하기**
   - 모르는 로그인 기록, 이상한 접속 알림, 저장된 화면 캡처를 남기세요.
   - 왜 필요한가요?  
     나중에 상담하거나 신고할 때 도움이 됩니다.

4. **웹캠 앱에서 연결된 기기 확인하기**
   - 모르는 기기나 낯선 접속이 있으면 바로 끊으세요.
   - 왜 필요한가요?  
     외부에서 계속 보고 있을 가능성을 줄일 수 있습니다.

## 이런 경우는 특히 중요합니다
- **모르는 로그인 기록이 보인다**
- **웹캠이 꺼졌는데도 이상하게 작동한 흔적이 있다**
- **영상이 외부로 나간 것 같은 알림이 있다**

이런 경우는 **해킹 의심**으로 보고, 빠르게 상담받는 것이 좋습니다.

## 상담 전에 준비하면 좋은 것
- 이상한 로그인 화면 캡처
- 알림 메시지 저장
- 영상 유출이 의심된 시간
- 사용 중인 웹캠 앱 이름
- 연결된 이메일 계정 정보

## 한마디로 정리하면
**118에 먼저 상담하고, 비밀번호를 바꾸고, 접속 기록을 남기는 것**이 가장 중요합니다.

원하시면 제가 **“웹캠 유출 의심 시 바로 하는 5단계”**로 더 쉽게 정리해드릴게요.  
더 궁금한 점이 있으면 언제든 물어보세요.


질문: 홈가전 IoT 기기의 보안 점검 항목을 알려주세요.

[1] SQL 생성 중...
    SELECT
    c.home_control_id,
    c.control_name,
    c.control_category,
    c.applicable_scope,
    c.hardware_or_security,
    c.implementation_summary,
    c.easy_explanation
FROM "2_child_home_iot_controls" AS c
ORDER BY c.page_start, c.home_control_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


...

랜섬웨어에 걸렸다면 **먼저 기기를 네트워크에서 분리하고**, **KISA 보호나라·KrCERT/CC에 신고**하는 것이 좋습니다.

## 어디에 신고하나요?
- **기관명:** KISA 보호나라·KrCERT/CC  
- **소속:** 한국인터넷진흥원(KISA)  
- **전화번호:** **118**  
- **연락 방법:** **온라인 침해사고 신고 / 전화상담**

이 기관은 **랜섬웨어, DDoS, 해킹 같은 침해사고 신고와 기술지원**을 담당합니다.

## 랜섬웨어일 때 보이는 증상
- **IoT 서버, PC, NAS의 파일이 암호화됨**
- **금전을 요구함**

이런 경우는 단순 오류가 아니라 **침해사고**로 볼 수 있습니다.

## 지금 바로 할 일
1. **장비를 인터넷과 공유기에서 분리하세요.**  
   - 왜 중요하냐면, 감염이 다른 기기로 퍼지는 것을 막기 위해서입니다.

2. **랜섬웨어가 걸린 화면과 메시지를 사진으로 남기세요.**  
   - 왜 중요하냐면, 신고할 때 상황을 설명하는 데 도움이 됩니다.

3. **파일을 함부로 지우거나 재설치하지 마세요.**  
   - 왜 중요하냐면, 원인 확인이나 복구에 필요한 정보가 사라질 수 있습니다.

4. **KISA 보호나라·KrCERT/CC로 신고하세요.**  
   - 전화 **118** 또는 **온라인 침해사고 신고**를 이용하세요.

## 신고할 때 준비하면 좋은 것
- 감염된 화면 사진
- 언제부터 이상했는지
- 어떤 기기인지  
  예: PC, NAS, 공유기, 웹캠 등
- 암호화되기 전후의 상황

## 긴급한 경우
- 여러 기기로 계속 퍼지는 것 같다면 **즉시 장비를 분리하고 바로 신고**하세요.
- 회사나 가족 여러 사람이 쓰는 장비라면 **더 빨리 대응**하는 것이 좋습니다.

현재 자료에서는 복구 방법이나 다른 기관 연락처까지는 정확하게 안내하기 어렵습니다.  
우선 **격리 후 118로 신고**하는 것이 가장 중요합니다.

더 궁금한 점이 있으면 언제든 물어보세요.


질문: IoT 공통 보안 가이드라인 중 인증과 관련된 항목을 알려주세요.

[1] SQL 생성 중...
    SELECT guideline_id, principle_name, lifecycle_stage, guideline_code, guideline_name, domain, description, easy_explanation
FROM "1_parent_iot_common_guidelines"
WHERE guideline_name ILIKE '%인증%'
   OR domain ILIKE '%인증%'
   OR description ILIKE '%인증%'
   OR easy_explanation ILIKE '%인증%'
ORDER BY guideline_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


조회 결과에서 **인증과 관련된 항목**은 다음입니다.

### 인증 관련 가이드라인
- **가이드라인 코드:** G02
- **원칙 이름:** 정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계
- **단계:** 설계·개발
- **가이드라인 이름:** 접근권한 관리·인증·종단간 통신 보안·데이터 암호화
- **도메인:** 인증_통신_암호화
- **설명:** IoT 서비스 환경에 맞는 접근권한, 인증, 통신 보호, 데이터 암호화 방안을 제공
- **쉬운 설명:** 기기가 다른 장치와 연결될 때 누가 접속할 수 있는지와 데이터 보호를 같이 보는 기준

### 쉽게 말하면
이 항목은 **“누가 기기에 들어올 수 있는지 확인하고, 연결 과정에서 정보가 새지 않게 보호하는 것”**을 뜻합니다.  
예를 들어 웹캠, 도어락, 공유기처럼 외부와 연결되는 기기는 **접속 권한 확인**과 **데이터 보호**가 중요합니다.

### 왜 중요한가요?
인증이 약하면:
- 모르는 사람이 기기에 접속할 수 있고
- 설정이 바뀌거나
- 영상, 문 잠금 정보, 네트워크 정보가 새어 나갈 수 있습니다.

현재 자료에서는 인증 관련 항목이 이 1개로 확인됩니다.  
더 궁금한 점이 있으면 언제든 물어보세요.


질문: 공유기가 갑자기 느려졌는데 해킹인지 확인하는 방법과 신고 절차를 알려주세요.

[1] SQL 생성 중...
    SELECT 
    a.agency_name,
    a.organization,
    a.phone,
    a.contact_method,
    a.main_role,
    i.incident_type,
    i.example_symptom,
    i.recommended_action
FROM "iot_security_incidents_child" i
JOIN "iot_security_agencies_parent" a
    ON i.agency_id = a.agency_id
WHERE (
    i.incident_type ILIKE '%해킹%'
    OR i.example_symptom ILIKE '%비정상%'
    OR i.example_symptom ILIKE '%로그인%'
    OR i.example_symptom ILIKE '%접속%'
)
OR a.main_role ILIKE '%침해사고%'
ORDER BY a.agency_id, i.incident_id
LIMIT 20;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


공유기가 갑자기 느려졌다면, **해킹이 아니라도** 일시적인 장애일 수 있습니다.  
하지만 **비정상적인 접속이나 대량 트래픽**이 함께 보이면 침해사고일 수 있습니다.

## 먼저 해킹인지 확인할 때 볼 것
아래를 순서대로 확인해 보세요.

1. **공유기 관리자 화면에 모르는 접속 기록이 있는지 확인**
   - 왜 중요한가요?  
     내가 모르는 사람이 관리자 계정에 들어갔다면 해킹 의심 신호입니다.
   - 이런 경우 주의하세요.  
     접속 시간, 접속한 기기 이름, 로그인 실패 기록이 이상하면 기록을 남겨두세요.

2. **인터넷이 느려진 시간에 다른 이상 증상이 같이 있었는지 확인**
   - 예를 들면  
     - 관리자 비밀번호가 바뀜  
     - 모르는 기기가 연결됨  
     - 설정이 바뀜  
   - 왜 중요한가요?  
     단순한 속도 저하보다 침해사고 가능성이 높아집니다.

3. **공유기 재부팅만 반복하지 말고 상태를 기록**
   - 왜 중요한가요?  
     로그와 화면을 남겨야 나중에 확인하거나 신고할 때 도움이 됩니다.
   - 저장할 것  
     - 느려진 시간  
     - 이상 증상 화면  
     - 접속기록  
     - 공유기 모델명

## 지금 바로 할 일
1. **공유기와 연결된 기기에서 모르는 로그인이나 접속이 있는지 확인**
2. **관리자 비밀번호와 Wi‑Fi 비밀번호를 바꾸기**
   - 왜 필요한가요?  
     누군가 들어왔을 수 있는 길을 바로 막을 수 있습니다.
3. **공유기 전원만 끄지 말고, 접속기록과 설정 화면을 먼저 보관하기**
   - 왜 필요한가요?  
     증거가 사라질 수 있습니다.
4. **공유기 펌웨어(기기 소프트웨어) 업데이트 확인**
   - 왜 필요한가요?  
     오래된 소프트웨어는 취약점이 남아 있을 수 있습니다.

## 신고는 어디로 하나요?
조회된 자료 기준으로는 **KISA 118 상담센터**와 **KISA 보호나라·KrCERT/CC**가 안내됩니다.

### 1) 먼저 상담이 필요한 경우
- **기관명:** KISA 118 상담센터  
- **전화번호:** 118  
- **연락 방법:** 전화 상담  
- **언제 적합한가요?**  
  공유기가 느려졌고, 해킹인지 헷갈리지만 확실하지 않을 때  
- **조회된 안내:**  
  “해킹 의심”이면 **우선 118에 상담 후 계정 비밀번호 변경 및 접속기록 보존**

### 2) 침해사고 신고가 필요한 경우
- **기관명:** KISA 보호나라·KrCERT/CC  
- **전화번호:** 118  
- **연락 방법:** 온라인 침해사고 신고 / 전화상담  
- **언제 적합한가요?**  
  - 모르는 로그인이나 비정상 접속이 확인될 때  
  - 공격자가 공유기나 관리페이지에 들어온 흔적이 있을 때  
  - 원인을 잘 모르겠지만 이상 징후가 계속될 때  
- **조회된 안내:**  
  그 밖의 해킹으로 신고하거나 118 상담을 이용하세요.

## 신고 전에 준비하면 좋은 것
- 접속기록 화면 사진
- 이상 증상이 나타난 시간
- 공유기 모델명
- 바뀐 설정이 있는지 메모
- 모르는 로그인이나 접속 기록

이 자료가 있으면 상담할 때 상황을 더 빨리 파악할 수 있습니다.

## 긴급한 경우
- **모르는 사람이 관리자 화면에 들어간 흔적이 분명함**
- **설정이 바뀌었고 계속 느려짐**
- **비정상 접속이 반복됨**

이런 경우는 **바로 118에 상담**하고, 필요하면 **보호나라·KrCERT/CC로 침해사고 신고**를 하세요.

현재 자료에서는 공유기 전용 신고 번호나 지역 기관은 정확하게 안내하기 어렵습니다.  
따라서 **118 상담**과 **보호나라 신고**를 우선 이용하는 것이 가장 맞습니다.

더 궁금한 점이 있으면 언제든 물어보세요.

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용